# Respiratory Disease Classification — Basic CNN + Log-Mel (ICBHI) — Notebook 2/4: **AdamW**

This notebook runs ONLY the **AdamW** experiments: 2 learning rates (0.001, 0.0001) × 5 seeds (42, 123, 2024, 7, 2025) = **10 runs**.
All other settings (preprocessing, features, patient-level split with `split_seed=42`, CNN, batch size 32, max 150 epochs, patience 15) are identical across the four notebooks,
so results are directly comparable. Because the split is deterministic given the same dataset, all four notebooks use exactly the same train/val/test patients.

**How to run on Kaggle:** add the dataset *vbookshelf/respiratory-sound-database*, enable a GPU, *Run All*. Finished runs are skipped if the session restarts.
Outputs go to `/kaggle/working/resp_cnn_adamw/` (CSVs in `results/`, figures in `figures/`, checkpoints in `runs/`).
Download `results/` and `runs/*/result.json` from all four notebooks if you want to compare optimizers across notebooks afterwards.

**Assumptions:** one recording = one sample (first 5 s, zero-padded); patient diagnosis = recording label; macro metrics average only over classes with support in the evaluated split; no class weighting / oversampling / augmentation.

## 1. Import Libraries and Environment Setup

In [ ]:
# ============================================================
# 1. IMPORT LIBRARIES AND ENVIRONMENT SETUP
# ============================================================
import os, sys, gc, json, time, copy, random, hashlib, platform
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # needed for deterministic CUDA matmuls

from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, confusion_matrix

# ---- Smoke-test switch: set env RESP_SMOKE_TEST=1 to run 1 run x 2 epochs into a separate folder ----
SMOKE_TEST = os.environ.get("RESP_SMOKE_TEST", "0") == "1"

# ---- Experiment configuration (exact settings requested) ----
CFG = dict(
    # audio / features
    sr=16000, duration=5.0, n_mels=128, n_fft=1024, hop_length=256, fmin=50, fmax=8000, top_db=80.0,
    # split
    split_seed=42, val_frac=0.15, test_frac=0.15,
    # training
    batch_size=32, max_epochs=150, patience=15, num_classes=8,
    optimizers=["AdamW"],
    learning_rates=[1e-3, 1e-4],
    seeds=[42, 123, 2024, 7, 2025],
)
if SMOKE_TEST:
    CFG.update(max_epochs=2, patience=2, optimizers=["AdamW"], learning_rates=[1e-3], seeds=[42])

# ---- Class definitions (fixed order = class index) ----
CLASS_NAMES = ["COPD", "Asthma", "Pneumonia", "Bronchiectasis", "Bronchiolitis", "URTI", "LRTI", "Healthy"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
# raw CSV label (lower-cased, stripped) -> target class. Anything not listed is reported and excluded.
LABEL_MAP = {"copd": "COPD", "asthma": "Asthma", "pneumonia": "Pneumonia", "bronchiectasis": "Bronchiectasis",
             "bronchiolitis": "Bronchiolitis", "urti": "URTI", "lrti": "LRTI", "healthy": "Healthy"}
assert CFG["num_classes"] == len(CLASS_NAMES)

# ---- Output directories (all under /kaggle/working when on Kaggle) ----
OUT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(os.environ.get("OUT_ROOT", "./working"))
OUT = OUT_ROOT / ("resp_cnn_adamw_smoke" if SMOKE_TEST else "resp_cnn_adamw")
CACHE_DIR, RUNS_DIR, FIG_DIR, RES_DIR, SPLIT_DIR = (OUT / d for d in ["cache", "runs", "figures", "results", "splits"])
for d in (CACHE_DIR, RUNS_DIR, FIG_DIR, RES_DIR, SPLIT_DIR):
    d.mkdir(parents=True, exist_ok=True)

def set_seed(seed: int):
    """Seed every RNG used in the pipeline."""
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

print("Python", platform.python_version(), "| torch", torch.__version__, "| librosa", librosa.__version__,
      "| numpy", np.__version__, "| pandas", pd.__version__)
print("Output directory:", OUT, "| SMOKE_TEST =", SMOKE_TEST)
print("Planned runs:", len(CFG["optimizers"]) * len(CFG["learning_rates"]) * len(CFG["seeds"]))

## 2. GPU Detection and Configuration

In [ ]:
# ============================================================
# 2. GPU DETECTION AND CONFIGURATION
# ============================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    n_gpu = torch.cuda.device_count()
    for i in range(n_gpu):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {p.name} | {p.total_memory / 1024**3:.1f} GB | compute capability {p.major}.{p.minor}")
    print("CUDA runtime:", torch.version.cuda, "| cuDNN:", torch.backends.cudnn.version())
else:
    print("WARNING: no GPU detected. Enable a GPU accelerator in the Kaggle notebook settings; CPU training will be very slow.")

# Reproducibility settings (bit-exact results are still not guaranteed across different GPU models / library versions)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)

## 3. Dataset Directory Inspection

In [ ]:
# ============================================================
# 3. DATASET DIRECTORY INSPECTION
# ============================================================
# Locate the audio folder and the diagnosis CSV automatically (nothing is hard-coded below /kaggle/input).
INPUT_ROOT = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path(os.environ.get("ICBHI_ROOT", "."))
print("Searching under:", INPUT_ROOT.resolve())

print("\nTop-level contents of the input directory:")
for p in sorted(INPUT_ROOT.iterdir()):
    print("  ", p.name)

all_wavs = list(INPUT_ROOT.rglob("*.wav"))
if not all_wavs:
    raise FileNotFoundError(f"No .wav files found under {INPUT_ROOT}. Did you add the ICBHI dataset to the notebook? "
                            "Please paste the output of `!ls -R /kaggle/input | head -50` so the paths can be fixed.")

# The audio directory is the folder that holds the most .wav files
dir_counts = Counter(p.parent for p in all_wavs)
AUDIO_DIR, n_in_dir = dir_counts.most_common(1)[0]
print(f"\nAudio directory : {AUDIO_DIR}  ({n_in_dir} wav files)")
if len(dir_counts) > 1:
    print("Other folders containing wav files (ignored):", {str(k): v for k, v in dir_counts.items() if k != AUDIO_DIR})
print("Annotation .txt files in the same folder:", len(list(AUDIO_DIR.glob("*.txt"))))

csv_candidates = sorted(INPUT_ROOT.rglob("*.csv"))
print("\nCSV files found:")
for c in csv_candidates:
    print("  ", c)
diag_candidates = [c for c in csv_candidates if c.name.lower() == "patient_diagnosis.csv"] or \
                  [c for c in csv_candidates if "diagnosis" in c.name.lower()]
if not diag_candidates:
    raise FileNotFoundError("patient_diagnosis.csv not found. Please share the dataset structure so the path can be set correctly.")
DIAG_CSV = diag_candidates[0]
print("\nDiagnosis CSV   :", DIAG_CSV)
print("Example wav names:", [p.name for p in sorted(all_wavs)[:3]])

## 4. Diagnosis CSV Inspection

In [ ]:
# ============================================================
# 4. DIAGNOSIS CSV INSPECTION
# ============================================================
print("--- first 5 raw lines of the file ---")
with open(DIAG_CSV, "r") as f:
    for _ in range(5):
        print(repr(f.readline()))

# The ICBHI file normally has NO header (two columns: patient id, diagnosis). Detect instead of assuming.
raw = pd.read_csv(DIAG_CSV, header=None)
first_cell = str(raw.iloc[0, 0]).strip()
if not first_cell.isdigit():
    print("\nFirst cell is not numeric -> treating first row as a header.")
    raw = pd.read_csv(DIAG_CSV, header=0)
else:
    print("\nNo header detected (first cell is numeric).")
print("Shape:", raw.shape, "| columns:", list(raw.columns))
assert raw.shape[1] >= 2, "Expected at least two columns (patient id, diagnosis)."

diag = raw.iloc[:, :2].copy()
diag.columns = ["patient_id", "diagnosis_raw"]
diag["patient_id"] = pd.to_numeric(diag["patient_id"], errors="coerce")
bad_ids = diag[diag["patient_id"].isna()]
if len(bad_ids):
    print(f"WARNING: {len(bad_ids)} diagnosis rows have a non-numeric patient id and are excluded:\n", bad_ids)
diag = diag.dropna(subset=["patient_id"]).copy()
diag["patient_id"] = diag["patient_id"].astype(int)
diag["diagnosis_raw"] = diag["diagnosis_raw"].astype(str).str.strip()

dups = diag[diag.duplicated("patient_id", keep=False)]
if len(dups):
    conflicting = dups.groupby("patient_id")["diagnosis_raw"].nunique()
    if (conflicting > 1).any():
        raise ValueError(f"Patients with conflicting diagnoses: {list(conflicting[conflicting > 1].index)}")
    print(f"Note: {dups['patient_id'].nunique()} duplicated patient ids with identical diagnoses -> de-duplicated.")
    diag = diag.drop_duplicates("patient_id")

diag["label"] = diag["diagnosis_raw"].str.lower().map(LABEL_MAP)
print("\nPatients in CSV:", len(diag))
print("\nRaw diagnosis values (patients per value):")
print(diag["diagnosis_raw"].value_counts().to_string())

unmapped = diag[diag["label"].isna()]
if len(unmapped):
    print(f"\nWARNING: {len(unmapped)} patients have diagnosis values outside the 8 target classes "
          f"({sorted(unmapped['diagnosis_raw'].unique())}). Their recordings will be excluded and logged.")
missing_classes = [c for c in CLASS_NAMES if c not in set(diag["label"].dropna())]
if missing_classes:
    raise RuntimeError(f"Dataset does not contain target classes: {missing_classes}. Stopping instead of inventing labels.")
print("\nAll 8 target classes are present in the diagnosis CSV.")

## 5. Audio File Loading

In [ ]:
# ============================================================
# 5. AUDIO FILE LOADING  (build the recording-level metadata table)
# ============================================================
# File names look like  101_1b1_Al_sc_Meditron.wav  ->  patientID_recordingIdx_chestLocation_acqMode_device
dropped_log = []   # every excluded recording is recorded here WITH a reason

rows = []
for p in sorted(AUDIO_DIR.glob("*.wav")):
    parts = p.stem.split("_")
    pid = int(parts[0]) if parts[0].isdigit() else None
    rows.append(dict(file=p.name, path=str(p), patient_id=pid,
                     chest_loc=parts[2] if len(parts) > 2 else None,
                     device=parts[4] if len(parts) > 4 else None))
meta = pd.DataFrame(rows)
print("Wav files:", len(meta))

# (a) patient id could not be parsed
bad = meta[meta["patient_id"].isna()]
dropped_log += [dict(file=f, reason="patient id not parsable from filename") for f in bad["file"]]
meta = meta.dropna(subset=["patient_id"]).copy()
meta["patient_id"] = meta["patient_id"].astype(int)

# (b) patient id not present in the diagnosis CSV
known = set(diag["patient_id"])
nodiag = meta[~meta["patient_id"].isin(known)]
dropped_log += [dict(file=f, reason="patient id missing from diagnosis CSV") for f in nodiag["file"]]
meta = meta[meta["patient_id"].isin(known)].copy()

meta = meta.merge(diag[["patient_id", "diagnosis_raw", "label"]], on="patient_id", how="left")

# (c) diagnosis outside the 8 target classes
unm = meta[meta["label"].isna()]
dropped_log += [dict(file=f, reason="diagnosis not in the 8 target classes") for f in unm["file"]]
meta = meta[meta["label"].notna()].copy().reset_index(drop=True)
meta["label_idx"] = meta["label"].map(CLASS_TO_IDX).astype(int)

no_audio = sorted(known - set(meta["patient_id"]))
print(f"Recordings matched to a target-class diagnosis: {len(meta)} | patients: {meta['patient_id'].nunique()}")
print(f"Patients in the CSV without any (usable) recording: {len(no_audio)} {no_audio[:20]}")
print(f"Recordings excluded so far: {len(dropped_log)}")
if dropped_log:
    print(pd.DataFrame(dropped_log)["reason"].value_counts().to_string())

# Final check that all 8 classes still exist after matching
still_missing = [c for c in CLASS_NAMES if c not in set(meta["label"])]
if still_missing:
    raise RuntimeError(f"After matching recordings to labels these classes have no audio: {still_missing}. Stopping.")
meta.head()

## 6. Audio Preprocessing

In [ ]:
# ============================================================
# 6. AUDIO PREPROCESSING
# ============================================================
TARGET_LEN = int(CFG["sr"] * CFG["duration"])   # 80,000 samples

def load_and_preprocess(path: str) -> np.ndarray:
    """Load -> mono -> resample to 16 kHz -> remove DC offset -> peak-normalise -> pad/truncate to 5 s.
    Raises ValueError for unreadable, empty, non-finite or silent audio (caller logs the reason)."""
    y, _ = librosa.load(path, sr=CFG["sr"], mono=True)           # librosa resamples to CFG['sr']
    if y.size == 0:
        raise ValueError("empty audio")
    if not np.all(np.isfinite(y)):
        raise ValueError("non-finite samples")
    y = y - np.mean(y)                                            # remove DC offset
    peak = np.max(np.abs(y))
    if peak < 1e-8:
        raise ValueError("silent audio")
    y = y / peak                                                  # peak normalisation to [-1, 1]
    if y.size >= TARGET_LEN:                                      # truncate (keep the first 5 s)
        y = y[:TARGET_LEN]
    else:                                                         # zero-pad at the end
        y = np.pad(y, (0, TARGET_LEN - y.size))
    return y.astype(np.float32)

# quick sanity check on one file
_y = load_and_preprocess(meta.loc[0, "path"])
print("Example:", meta.loc[0, "file"], "| shape:", _y.shape, "| min/max:", _y.min(), _y.max(), "| dtype:", _y.dtype)

## 7. Log-Mel Spectrogram Generation

In [ ]:
# ============================================================
# 7. LOG-MEL SPECTROGRAM GENERATION  (cached to disk)
# ============================================================
def logmel(y: np.ndarray) -> np.ndarray:
    """Power mel spectrogram -> decibel scale. Output shape: (n_mels, frames) = (128, 313)."""
    S = librosa.feature.melspectrogram(y=y, sr=CFG["sr"], n_fft=CFG["n_fft"], hop_length=CFG["hop_length"],
                                       n_mels=CFG["n_mels"], fmin=CFG["fmin"], fmax=CFG["fmax"], power=2.0)
    return librosa.power_to_db(S, ref=np.max, top_db=CFG["top_db"]).astype(np.float32)

# cache key depends on the feature config and the exact file list -> stale caches are never reused
feat_cfg = {k: CFG[k] for k in ["sr", "duration", "n_mels", "n_fft", "hop_length", "fmin", "fmax", "top_db"]}
sig = hashlib.md5((json.dumps(feat_cfg, sort_keys=True) + "|".join(meta["file"])).encode()).hexdigest()[:10]
CACHE_FILE = CACHE_DIR / f"logmel_{sig}.npz"
DROP_FILE = CACHE_DIR / f"dropped_{sig}.json"

if CACHE_FILE.exists():
    z = np.load(CACHE_FILE, allow_pickle=False)
    X_all, good_files = z["X"], list(z["files"])
    dropped_log = json.load(open(DROP_FILE))
    print(f"Loaded cached features: {X_all.shape} from {CACHE_FILE.name}")
else:
    feats, good_files = [], []
    for f, p in tqdm(list(zip(meta["file"], meta["path"])), desc="Log-Mel extraction"):
        try:
            feats.append(logmel(load_and_preprocess(p)))
            good_files.append(f)
        except Exception as e:                                    # corrupted / unreadable / silent file
            dropped_log.append(dict(file=f, reason=f"audio error: {type(e).__name__}: {e}"))
    X_all = np.stack(feats).astype(np.float32)
    np.savez(CACHE_FILE, X=X_all, files=np.array(good_files))
    json.dump(dropped_log, open(DROP_FILE, "w"))
    print(f"Computed and cached features: {X_all.shape} -> {CACHE_FILE.name}")

# keep metadata aligned with the feature array
meta = meta.set_index("file").loc[good_files].reset_index()
assert len(meta) == len(X_all)
dropped_df = pd.DataFrame(dropped_log, columns=["file", "reason"])
dropped_df.to_csv(RES_DIR / "excluded_recordings.csv", index=False)
print(f"\nUsable recordings: {len(meta)} | excluded: {len(dropped_df)}")
if len(dropped_df):
    print(dropped_df["reason"].value_counts().to_string())

# Visual sanity check
fig, ax = plt.subplots(figsize=(8, 3.5))
im = ax.imshow(X_all[0], origin="lower", aspect="auto", cmap="magma")
ax.set_title(f"Log-Mel spectrogram — {meta.loc[0, 'file']} ({meta.loc[0, 'label']})")
ax.set_xlabel("Frame"); ax.set_ylabel("Mel band"); fig.colorbar(im, ax=ax, label="dB")
fig.tight_layout(); fig.savefig(FIG_DIR / "example_logmel.png", dpi=150); plt.show()

## 8. Dataset and Class Distribution Analysis

In [ ]:
# ============================================================
# 8. DATASET AND CLASS DISTRIBUTION ANALYSIS
# ============================================================
dist = pd.DataFrame({
    "recordings": meta["label"].value_counts().reindex(CLASS_NAMES).fillna(0).astype(int),
    "patients": meta.groupby("label")["patient_id"].nunique().reindex(CLASS_NAMES).fillna(0).astype(int),
})
dist["recordings_%"] = (100 * dist["recordings"] / dist["recordings"].sum()).round(1)
print("Class distribution BEFORE splitting")
display(dist)
print(f"Total: {dist['recordings'].sum()} recordings from {dist['patients'].sum()} patients")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
dist["recordings"].plot.bar(ax=axes[0], color="steelblue"); axes[0].set_title("Recordings per class")
dist["patients"].plot.bar(ax=axes[1], color="darkorange");  axes[1].set_title("Patients per class")
for a in axes: a.set_ylabel("Count"); a.set_xlabel("Class"); a.tick_params(axis="x", rotation=45)
fig.tight_layout(); fig.savefig(FIG_DIR / "class_distribution.png", dpi=150); plt.show()

few = dist[dist["patients"] < 3]
if len(few):
    print("NOTE: classes with fewer than 3 patients cannot appear in train, validation AND test at the same time "
          "under a patient-independent split:\n", few[["patients"]].to_string())

## 9. Patient-Independent Data Splitting

In [ ]:
# ============================================================
# 9. PATIENT-INDEPENDENT DATA SPLITTING (fixed for all 40 runs)
# ============================================================
# Method: each patient has exactly one diagnosis, so the split is done at the PATIENT level and allocated
# class-by-class (i.e. stratified by patient diagnosis). For a class with n patients:
#   n >= 3 : n_val = n_test = max(1, round(0.15 n)), remainder -> train
#   n == 2 : 1 train, 0 val, 1 test      n == 1 : 1 train
# Percentages are therefore approximate at the recording level (patients differ greatly in recording count).
def patient_split(patients_df, seed, val_frac, test_frac):
    rng = np.random.RandomState(seed)
    assign = {}
    for label, grp in patients_df.groupby("label"):
        ids = np.array(sorted(grp["patient_id"]))
        rng.shuffle(ids)
        n = len(ids)
        if n >= 3:
            n_val = max(1, int(np.floor(val_frac * n + 0.5)))
            n_test = max(1, int(np.floor(test_frac * n + 0.5)))
        elif n == 2:
            n_val, n_test = 0, 1
        else:
            n_val, n_test = 0, 0
        for i, pid in enumerate(ids):
            assign[int(pid)] = "test" if i < n_test else ("val" if i < n_test + n_val else "train")
    return assign

patients = meta.drop_duplicates("patient_id")[["patient_id", "label"]]
assert patients["patient_id"].is_unique
assign = patient_split(patients, CFG["split_seed"], CFG["val_frac"], CFG["test_frac"])
meta["split"] = meta["patient_id"].map(assign)
assert meta["split"].notna().all()

# ---- Leakage check ----
ids = {s: set(meta.loc[meta["split"] == s, "patient_id"]) for s in ["train", "val", "test"]}
assert not (ids["train"] & ids["val"]) and not (ids["train"] & ids["test"]) and not (ids["val"] & ids["test"]), "PATIENT LEAKAGE!"
print("Leakage check passed: no patient appears in more than one split.\n")

for s in ["train", "val", "test"]:
    n_rec = int((meta["split"] == s).sum())
    print(f"{s.upper():5s}: {len(ids[s]):3d} patients ({100*len(ids[s])/len(patients):.1f}% of patients) | "
          f"{n_rec:4d} recordings ({100*n_rec/len(meta):.1f}% of recordings)")
    print("       patient IDs:", sorted(ids[s]))

pt_dist = pd.crosstab(meta.drop_duplicates("patient_id")["label"], meta.drop_duplicates("patient_id")["split"]).reindex(index=CLASS_NAMES, columns=["train", "val", "test"]).fillna(0).astype(int)
rec_dist = pd.crosstab(meta["label"], meta["split"]).reindex(index=CLASS_NAMES, columns=["train", "val", "test"]).fillna(0).astype(int)
print("\nPatients per class per split"); display(pt_dist)
print("Recordings per class per split"); display(rec_dist)

absent = {s: [c for c in CLASS_NAMES if rec_dist.loc[c, s] == 0] for s in ["train", "val", "test"]}
for s, a in absent.items():
    print(f"Classes absent from {s}: {a if a else 'none'}")
print("\nMetrics for classes absent from the validation / test set are reported as undefined; macro averages use only classes with support.")

meta[["file", "patient_id", "label", "split"]].to_csv(SPLIT_DIR / "split_assignment.csv", index=False)
pt_dist.to_csv(SPLIT_DIR / "patients_per_class_per_split.csv"); rec_dist.to_csv(SPLIT_DIR / "recordings_per_class_per_split.csv")

## 10. PyTorch Dataset and DataLoader

In [ ]:
# ============================================================
# 10. PYTORCH DATASET AND DATALOADER
# ============================================================
# Features live in CPU RAM (~150 MB); only mini-batches are moved to the GPU.
train_mask = (meta["split"] == "train").values
# Normalisation statistics come from the TRAINING split only (no val/test leakage)
FEAT_MEAN, FEAT_STD = float(X_all[train_mask].mean()), float(X_all[train_mask].std() + 1e-8)
print(f"Train-set feature mean={FEAT_MEAN:.3f}, std={FEAT_STD:.3f}")

X_t = torch.from_numpy((X_all - FEAT_MEAN) / FEAT_STD).unsqueeze(1)          # (N, 1, 128, 313)
y_t = torch.from_numpy(meta["label_idx"].values).long()
SPLIT_IDX = {s: torch.from_numpy(np.where(meta["split"].values == s)[0]) for s in ["train", "val", "test"]}
SPLIT_DS = {s: TensorDataset(X_t[idx], y_t[idx]) for s, idx in SPLIT_IDX.items()}
INPUT_SHAPE = tuple(X_t.shape[1:])
print("Input shape per sample:", INPUT_SHAPE, "| samples:", {s: len(d) for s, d in SPLIT_DS.items()})

def make_loader(split: str, seed: int, shuffle: bool = False) -> DataLoader:
    """Deterministic loader: shuffling order depends only on the run seed."""
    g = torch.Generator(); g.manual_seed(seed)
    return DataLoader(SPLIT_DS[split], batch_size=CFG["batch_size"], shuffle=shuffle, generator=g,
                      num_workers=0, pin_memory=(DEVICE.type == "cuda"), drop_last=False)

## 11. Basic CNN Architecture

In [ ]:
# ============================================================
# 11. BASIC CNN ARCHITECTURE
# ============================================================
class BasicCNN(nn.Module):
    def __init__(self, n_classes: int = 8, channels=(32, 64, 128, 256), fc_dim: int = 128, dropout: float = 0.5):
        super().__init__()
        blocks, in_ch = [], 1
        for out_ch in channels:                                    # 4 x (Conv + BN + ReLU + MaxPool)
            blocks += [nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
                       nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True), nn.MaxPool2d(2)]
            in_ch = out_ch
        self.features = nn.Sequential(*blocks)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(in_ch, fc_dim), nn.ReLU(inplace=True),
                                        nn.Dropout(dropout), nn.Linear(fc_dim, n_classes))
    def forward(self, x):
        return self.classifier(self.pool(self.features(x)))

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

_m = BasicCNN(CFG["num_classes"])
print(_m)
print(f"\nTrainable parameters: {count_params(_m):,}")
with torch.no_grad():
    print("Output shape for a dummy batch:", tuple(_m.eval()(torch.zeros(2, *INPUT_SHAPE)).shape))
del _m

## 12. Training and Validation Functions

In [ ]:
# ============================================================
# 12. TRAINING AND VALIDATION FUNCTIONS
# ============================================================
def make_optimizer(name: str, params, lr: float):
    if name == "Adam":    return torch.optim.Adam(params, lr=lr)
    if name == "AdamW":   return torch.optim.AdamW(params, lr=lr)
    if name == "SGD":     return torch.optim.SGD(params, lr=lr, momentum=0.9)
    if name == "RMSprop": return torch.optim.RMSprop(params, lr=lr)
    raise ValueError(name)

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        correct += (out.argmax(1) == yb).sum().item()
        n += xb.size(0)
    return total_loss / n, correct / n

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, n = 0.0, 0
    ys, preds, probs = [], [], []
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
        out = model(xb)
        total_loss += criterion(out, yb).item() * xb.size(0)
        n += xb.size(0)
        ys.append(yb.cpu()); preds.append(out.argmax(1).cpu()); probs.append(torch.softmax(out, 1).cpu())
    y = torch.cat(ys).numpy(); p = torch.cat(preds).numpy(); pr = torch.cat(probs).numpy()
    return total_loss / n, float((y == p).mean()), y, p, pr

def to_cpu(obj):
    """Detached CPU copy of (nested) tensors, used to snapshot the best checkpoint."""
    if torch.is_tensor(obj): return obj.detach().cpu().clone()
    if isinstance(obj, dict): return {k: to_cpu(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)): return type(obj)(to_cpu(v) for v in obj)
    return copy.deepcopy(obj)

## 13. Early Stopping Implementation

In [ ]:
# ============================================================
# 13. EARLY STOPPING (monitors validation Macro-F1, mode = max)
# ============================================================
class EarlyStopping:
    """Stops when the monitored score has not strictly improved for `patience` consecutive epochs.
    Keeps an in-memory snapshot (model + optimizer state) of the best epoch."""
    def __init__(self, patience: int):
        self.patience, self.best_score, self.best_epoch = patience, -np.inf, 0
        self.counter, self.should_stop = 0, False
        self.best_model_state, self.best_opt_state = None, None
    def step(self, score, epoch, model, optimizer) -> bool:
        if score > self.best_score:
            self.best_score, self.best_epoch, self.counter = score, epoch, 0
            self.best_model_state, self.best_opt_state = to_cpu(model.state_dict()), to_cpu(optimizer.state_dict())
            return True
        self.counter += 1
        self.should_stop = self.counter >= self.patience
        return False

## 14. Evaluation Metrics (and the single-experiment runner)

In [ ]:
# ============================================================
# 14. EVALUATION METRICS + EXPERIMENT RUNNER
# ============================================================
def compute_metrics(y_true, y_pred, y_prob=None, n_classes=8):
    """All metrics derived from the 8x8 confusion matrix.
    - Class-wise precision is undefined (NaN) if the class was never predicted; recall is undefined if the class has no true samples.
    - Macro averages use ONLY classes with support>0 in y_true (precision of a supported-but-never-predicted class counts as 0).
    - Macro AUC (one-vs-rest) averages the per-class AUCs that are mathematically defined (class present AND at least one other class present)."""
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n_classes)))
    tp = np.diag(cm).astype(float); sup = cm.sum(1).astype(float); pc = cm.sum(0).astype(float)
    fp, fn = pc - tp, sup - tp
    with np.errstate(divide="ignore", invalid="ignore"):
        prec = np.where(pc > 0, tp / pc, np.nan)
        rec = np.where(sup > 0, tp / sup, np.nan)
        den = 2 * tp + fp + fn
        f1 = np.where(den > 0, 2 * tp / den, np.nan)
    present = sup > 0
    out = dict(accuracy=float(tp.sum() / cm.sum()),
               macro_precision=float(np.mean(np.nan_to_num(prec[present], nan=0.0))),
               macro_recall=float(np.mean(rec[present])),
               macro_f1=float(np.mean(f1[present])),
               n_classes_evaluated=int(present.sum()), cm=cm, prec=prec, rec=rec, f1=f1, support=sup)
    aucs = np.full(n_classes, np.nan)
    if y_prob is not None:
        for c in range(n_classes):
            pos = (np.asarray(y_true) == c)
            if 0 < pos.sum() < len(pos):
                aucs[c] = roc_auc_score(pos, y_prob[:, c])
    out["auc_per_class"] = aucs
    out["macro_auc"] = float(np.nanmean(aucs)) if np.isfinite(aucs).any() else float("nan")
    out["n_auc_classes"] = int(np.isfinite(aucs).sum())
    return out

def run_experiment(opt_name: str, lr: float, seed: int) -> dict:
    run_id = f"{opt_name}_lr{lr:g}_seed{seed}"
    rdir = RUNS_DIR / run_id
    if (rdir / "result.json").exists():                          # resume support: skip finished runs
        return json.load(open(rdir / "result.json"))
    rdir.mkdir(parents=True, exist_ok=True)

    set_seed(seed)
    model = BasicCNN(CFG["num_classes"]).to(DEVICE)              # fresh model for every run
    optimizer = make_optimizer(opt_name, model.parameters(), lr)
    criterion = nn.CrossEntropyLoss()                            # no class weights (nothing unreported)
    tr_loader, va_loader, te_loader = make_loader("train", seed, True), make_loader("val", seed), make_loader("test", seed)
    stopper, history = EarlyStopping(CFG["patience"]), []

    t0 = time.time()
    for epoch in range(1, CFG["max_epochs"] + 1):
        tr_loss, tr_acc = train_one_epoch(model, tr_loader, optimizer, criterion)
        va_loss, va_acc, yv, pv, _ = evaluate(model, va_loader, criterion)
        vm = compute_metrics(yv, pv, None, CFG["num_classes"])
        history.append(dict(optimizer=opt_name, learning_rate=lr, seed=seed, epoch=epoch,
                            train_loss=tr_loss, train_acc=tr_acc, val_loss=va_loss, val_acc=va_acc, val_macro_f1=vm["macro_f1"]))
        stopper.step(vm["macro_f1"], epoch, model, optimizer)
        if stopper.should_stop: break
    train_time = time.time() - t0

    # ---- restore best checkpoint (selected on validation Macro-F1) and evaluate ONCE on the held-out test set ----
    model.load_state_dict(stopper.best_model_state)
    te_loss, _, yt, pt, prt = evaluate(model, te_loader, criterion)
    tm = compute_metrics(yt, pt, prt, CFG["num_classes"])
    best_val_row = history[stopper.best_epoch - 1]

    torch.save(dict(model_state_dict=stopper.best_model_state, optimizer_state_dict=stopper.best_opt_state,
                    epoch=stopper.best_epoch, best_val_macro_f1=stopper.best_score, optimizer=opt_name,
                    learning_rate=lr, seed=seed, class_to_idx=CLASS_TO_IDX, feat_mean=FEAT_MEAN, feat_std=FEAT_STD),
               rdir / "best.pt")
    pd.DataFrame(history).to_csv(rdir / "history.csv", index=False)
    pd.DataFrame(dict(optimizer=opt_name, learning_rate=lr, seed=seed, **{"class": CLASS_NAMES}, support=tm["support"].astype(int),
                      precision=tm["prec"], recall=tm["rec"], f1=tm["f1"], auc=tm["auc_per_class"])).to_csv(rdir / "classwise.csv", index=False)
    np.save(rdir / "confusion_matrix.npy", tm["cm"])

    result = dict(run_id=run_id, optimizer=opt_name, learning_rate=lr, seed=seed,
                  best_epoch=stopper.best_epoch, epochs_run=len(history), stopped_early=bool(stopper.should_stop),
                  best_val_macro_f1=float(stopper.best_score), val_accuracy_at_best=float(best_val_row["val_acc"]),
                  test_accuracy=tm["accuracy"], test_macro_precision=tm["macro_precision"], test_macro_recall=tm["macro_recall"],
                  test_macro_f1=tm["macro_f1"], test_macro_auc=tm["macro_auc"], test_loss=float(te_loss),
                  n_test_classes_evaluated=tm["n_classes_evaluated"], n_auc_classes=tm["n_auc_classes"],
                  training_time_s=float(train_time), n_params=count_params(model))
    json.dump(result, open(rdir / "result.json", "w"), indent=1)      # written last = marks the run as complete

    del model, optimizer, tr_loader, va_loader, te_loader, stopper
    gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    return result

def run_optimizer_block(opt_name: str):
    """Runs (or skips if finished) every learning-rate x seed combination for one optimizer."""
    jobs = [(lr, s) for lr in CFG["learning_rates"] for s in CFG["seeds"]]
    results = []
    for lr, s in tqdm(jobs, desc=f"{opt_name} runs"):
        r = run_experiment(opt_name, lr, s)
        results.append(r)
        tqdm.write(f"{r['run_id']:28s} best_ep={r['best_epoch']:3d}/{r['epochs_run']:3d} val_F1={r['best_val_macro_f1']:.4f} "
                   f"| test acc={r['test_accuracy']:.4f} F1={r['test_macro_f1']:.4f} AUC={r['test_macro_auc']:.4f} | {r['training_time_s']:.0f}s")
    return pd.DataFrame(results)

## 15. AdamW Experiments

In [ ]:
# ============================================================
# 15. ADAMW EXPERIMENTS  (2 learning rates x 5 seeds = 10 runs; finished runs are skipped)
# ============================================================
res_adamw = run_optimizer_block("AdamW")
display(res_adamw[["run_id", "best_epoch", "best_val_macro_f1", "test_accuracy", "test_macro_f1", "test_macro_auc"]])

## 16. Confusion Matrices and Training Curves

In [ ]:
# ============================================================
# 16. CONFUSION MATRICES AND TRAINING CURVES
# ============================================================
def load_all_results() -> pd.DataFrame:
    rows = [json.load(open(p)) for p in sorted(RUNS_DIR.glob("*/result.json"))]
    return pd.DataFrame(rows)

results_df = load_all_results()
expected = len(CFG["optimizers"]) * len(CFG["learning_rates"]) * len(CFG["seeds"])
print(f"Completed runs found: {len(results_df)} / {expected}")
if len(results_df) < expected:
    print("WARNING: some runs are missing - re-run section 15 to finish them.")

def plot_confusion(cm, title, path, show=False):
    support = cm.sum(1)
    norm = np.divide(cm, support[:, None], out=np.zeros(cm.shape, dtype=float), where=support[:, None] > 0)
    fig, ax = plt.subplots(figsize=(8.5, 7))
    im = ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(CLASS_NAMES))); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
    ax.set_yticks(range(len(CLASS_NAMES))); ax.set_yticklabels([f"{c} (n={int(s)})" for c, s in zip(CLASS_NAMES, support)])
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center", color="white" if norm[i, j] > 0.5 else "black")
    ax.set_xlabel("Predicted label"); ax.set_ylabel("True label (n = test recordings)")
    ax.set_title(title); fig.colorbar(im, ax=ax, label="Row-normalised fraction")
    fig.tight_layout(); fig.savefig(path, dpi=150)
    if show: plt.show()
    plt.close(fig)

def plot_curves(hist: pd.DataFrame, title, path, show=False):
    fig, axes = plt.subplots(2, 2, figsize=(11, 7))
    spec = [("train_loss", "Training Loss vs Epoch", "Loss"), ("val_loss", "Validation Loss vs Epoch", "Loss"),
            ("train_acc", "Training Accuracy vs Epoch", "Accuracy"), ("val_acc", "Validation Accuracy vs Epoch", "Accuracy")]
    best_ep = hist.loc[hist["val_macro_f1"].idxmax(), "epoch"]
    for ax, (col, ttl, yl) in zip(axes.ravel(), spec):
        ax.plot(hist["epoch"], hist[col], label=col.replace("_", " "))
        ax.axvline(best_ep, color="red", ls="--", alpha=0.6, label=f"best epoch ({int(best_ep)})")
        ax.set_title(ttl); ax.set_xlabel("Epoch"); ax.set_ylabel(yl); ax.legend(); ax.grid(alpha=0.3)
    fig.suptitle(title); fig.tight_layout(); fig.savefig(path, dpi=150)
    if show: plt.show()
    plt.close(fig)

# Save a confusion matrix and a curve figure for EVERY run (cheap), display only the best run per optimizer.
# "Best" = highest VALIDATION Macro-F1 (never test).
best_run_ids = results_df.loc[results_df.groupby("optimizer")["best_val_macro_f1"].idxmax(), "run_id"].tolist()
(FIG_DIR / "confusion_matrices").mkdir(exist_ok=True); (FIG_DIR / "training_curves").mkdir(exist_ok=True)
for r in tqdm(results_df.to_dict("records"), desc="Saving figures"):
    rdir = RUNS_DIR / r["run_id"]
    show = r["run_id"] in best_run_ids
    ttl = f"{r['optimizer']} lr={r['learning_rate']:g} seed={r['seed']}"
    plot_confusion(np.load(rdir / "confusion_matrix.npy"),
                   f"Confusion matrix (test) — {ttl}\nacc={r['test_accuracy']:.3f}, macro-F1={r['test_macro_f1']:.3f}",
                   FIG_DIR / "confusion_matrices" / f"cm_{r['run_id']}.png", show=show)
    plot_curves(pd.read_csv(rdir / "history.csv"), f"Training curves — {ttl}",
                FIG_DIR / "training_curves" / f"curves_{r['run_id']}.png", show=show)
print("Displayed: best run per optimizer by validation Macro-F1 ->", best_run_ids)

## 17. CSV Results and Summary

In [ ]:
# ============================================================
# 17. CSV RESULTS AND SUMMARY
# ============================================================
results_df = load_all_results().sort_values(["optimizer", "learning_rate", "seed"]).reset_index(drop=True)

# ---- (1) main results: one row per run ----
main_cols = {"optimizer": "Optimizer", "learning_rate": "Learning Rate", "seed": "Seed", "best_epoch": "Best Epoch",
             "best_val_macro_f1": "Best Val Macro-F1", "test_accuracy": "Accuracy", "test_macro_precision": "Macro Precision",
             "test_macro_recall": "Macro Recall", "test_macro_f1": "Macro-F1", "test_macro_auc": "Macro AUC",
             "training_time_s": "Training Time (s)", "n_params": "Number of Parameters",
             "epochs_run": "Epochs Run", "n_test_classes_evaluated": "Test Classes With Support", "n_auc_classes": "Classes In Macro AUC"}
main_df = results_df[list(main_cols)].rename(columns=main_cols)
main_df.to_csv(RES_DIR / "main_results.csv", index=False)

# ---- (2) per-epoch history, (3) class-wise reports for all runs ----
hist_all = pd.concat([pd.read_csv(p) for p in sorted(RUNS_DIR.glob("*/history.csv"))], ignore_index=True)
hist_all.to_csv(RES_DIR / "training_history_all_runs.csv", index=False)
cw_all = pd.concat([pd.read_csv(p) for p in sorted(RUNS_DIR.glob("*/classwise.csv"))], ignore_index=True)
cw_all.to_csv(RES_DIR / "classwise_report_all_runs.csv", index=False)

# ---- (4) mean +/- SD per optimizer x learning rate (sample SD, ddof=1, across seeds) ----
metric_cols = ["best_val_macro_f1", "test_accuracy", "test_macro_precision", "test_macro_recall", "test_macro_f1",
               "test_macro_auc", "best_epoch", "training_time_s"]
g = results_df.groupby(["optimizer", "learning_rate"])
summary = g[metric_cols].agg(["mean", "std"])
summary.columns = [f"{m}_{s}" for m, s in summary.columns]
summary["n_seeds"] = g.size()
summary = summary.reset_index()
for m in metric_cols:
    summary[f"{m}_mean±sd"] = summary.apply(lambda r: f"{r[m + '_mean']:.4f} ± {r[m + '_std']:.4f}", axis=1)
summary.to_csv(RES_DIR / "summary_mean_std_per_optimizer_lr.csv", index=False)

# ---- (5) class-wise mean +/- SD across seeds for each optimizer x lr ----
cw_summary = cw_all.groupby(["optimizer", "learning_rate", "class"])[["precision", "recall", "f1", "auc"]].agg(["mean", "std"])
cw_summary.columns = [f"{m}_{s}" for m, s in cw_summary.columns]
cw_summary.reset_index().to_csv(RES_DIR / "classwise_summary_mean_std.csv", index=False)

print("Saved CSVs in", RES_DIR); [print("  ", p.name) for p in sorted(RES_DIR.glob("*.csv"))]
print("\nMain results (test set, per run):")
display(main_df)
print("Mean ± SD across seeds:")
display(summary[["optimizer", "learning_rate", "n_seeds"] + [f"{m}_mean±sd" for m in ["best_val_macro_f1", "test_accuracy", "test_macro_f1", "test_macro_auc"]]])

## 18. Final Results for AdamW (learning-rate comparison)

In [ ]:
# ============================================================
# 18. FINAL RESULTS FOR ADAMW
# ============================================================
# Best configuration is chosen ONLY from validation Macro-F1 (mean over seeds); test performance is then reported for it.
summary_sorted = summary.sort_values("best_val_macro_f1_mean", ascending=False).reset_index(drop=True)
best_cfg = summary_sorted.iloc[0]

# Per optimizer: choose the learning rate with the higher mean validation Macro-F1
opt_best = summary.loc[summary.groupby("optimizer")["best_val_macro_f1_mean"].idxmax()].sort_values("best_val_macro_f1_mean", ascending=False)
comparison = opt_best[["optimizer", "learning_rate", "n_seeds",
                       "best_val_macro_f1_mean", "best_val_macro_f1_std",
                       "test_accuracy_mean", "test_accuracy_std", "test_macro_precision_mean", "test_macro_precision_std",
                       "test_macro_recall_mean", "test_macro_recall_std", "test_macro_f1_mean", "test_macro_f1_std",
                       "test_macro_auc_mean", "test_macro_auc_std", "training_time_s_mean"]].reset_index(drop=True)
comparison.to_csv(RES_DIR / "final_optimizer_comparison.csv", index=False)
print("Results for AdamW (best LR by validation Macro-F1; test metrics = mean ± SD over seeds)")
display(comparison.round(4))

print("\n=== Best configuration (selected by mean validation Macro-F1) ===")
print(f"Optimizer={best_cfg['optimizer']}, LR={best_cfg['learning_rate']:g}")
print(f"  Validation Macro-F1 : {best_cfg['best_val_macro_f1_mean']:.4f} ± {best_cfg['best_val_macro_f1_std']:.4f}")
for lab, k in [("Accuracy", "test_accuracy"), ("Macro Precision", "test_macro_precision"), ("Macro Recall", "test_macro_recall"),
               ("Macro-F1", "test_macro_f1"), ("Macro AUC", "test_macro_auc")]:
    print(f"  Held-out test {lab:16s}: {best_cfg[k + '_mean']:.4f} ± {best_cfg[k + '_std']:.4f}")

# ---- Plots ----
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
opts, lrs = CFG["optimizers"], CFG["learning_rates"]
w = 0.8 / len(lrs); x = np.arange(len(opts))
for k, lr in enumerate(lrs):                                              # (a) mean ± SD test Macro-F1
    sub = summary[summary["learning_rate"] == lr].set_index("optimizer").reindex(opts)
    axes[0].bar(x + k * w, sub["test_macro_f1_mean"], w, yerr=sub["test_macro_f1_std"], capsize=4, label=f"lr={lr:g}")
axes[0].set_xticks(x + w * (len(lrs) - 1) / 2); axes[0].set_xticklabels(opts)
axes[0].set_title("Test Macro-F1 (mean ± SD over seeds)"); axes[0].set_ylabel("Macro-F1"); axes[0].set_xlabel("Optimizer"); axes[0].legend()

labels, data = [], []                                                      # (b) per-seed distribution
for o in opts:
    for lr in lrs:
        d = results_df[(results_df["optimizer"] == o) & (results_df["learning_rate"] == lr)]["test_macro_f1"].values
        if len(d): labels.append(f"{o}\n{lr:g}"); data.append(d)
axes[1].boxplot(data, showmeans=True); axes[1].set_xticks(range(1, len(labels) + 1)); axes[1].set_xticklabels(labels)
for i, d in enumerate(data, 1): axes[1].scatter(np.full(len(d), i), d, color="k", s=12, zorder=3)
axes[1].set_title("Test Macro-F1 per seed"); axes[1].set_ylabel("Macro-F1"); axes[1].set_xlabel("Optimizer / learning rate")

for k, lr in enumerate(lrs):                                              # (c) validation vs test (selection vs generalisation)
    sub = summary[summary["learning_rate"] == lr].set_index("optimizer").reindex(opts)
    axes[2].scatter(sub["best_val_macro_f1_mean"], sub["test_macro_f1_mean"], s=70, label=f"lr={lr:g}")
    for o in opts:
        if o in sub.index and np.isfinite(sub.loc[o, "test_macro_f1_mean"]):
            axes[2].annotate(o, (sub.loc[o, "best_val_macro_f1_mean"], sub.loc[o, "test_macro_f1_mean"]), fontsize=8, xytext=(3, 3), textcoords="offset points")
axes[2].set_title("Validation vs test Macro-F1 (mean)"); axes[2].set_xlabel("Validation Macro-F1"); axes[2].set_ylabel("Test Macro-F1"); axes[2].legend()
fig.tight_layout(); fig.savefig(FIG_DIR / "optimizer_comparison.png", dpi=150); plt.show()

# ---- Objective, data-driven reading of the results (no claims of significance) ----
print("\n=== Objective summary (generated from the numbers above) ===")
ranked = summary_sorted[["optimizer", "learning_rate", "best_val_macro_f1_mean", "test_macro_f1_mean", "test_macro_f1_std"]]
for i, r in ranked.iterrows():
    print(f"{i+1}. {r['optimizer']:8s} lr={r['learning_rate']:<7g} val F1={r['best_val_macro_f1_mean']:.3f} | test F1={r['test_macro_f1_mean']:.3f} ± {r['test_macro_f1_std']:.3f}")
top, bottom = ranked.iloc[0], ranked.iloc[-1]
print(f"\n- Spread of mean test Macro-F1 across configurations: {ranked['test_macro_f1_mean'].min():.3f} – {ranked['test_macro_f1_mean'].max():.3f}.")
if len(ranked) > 1 and ranked.iloc[1]["test_macro_f1_mean"] + ranked.iloc[1]["test_macro_f1_std"] >= top["test_macro_f1_mean"] - top["test_macro_f1_std"]:
    print("- The ±1 SD ranges of the top two configurations overlap, so their order should be treated as unresolved.")
for lr in lrs:
    print(f"- Mean test Macro-F1 over all optimizers at lr={lr:g}: {summary[summary['learning_rate'] == lr]['test_macro_f1_mean'].mean():.3f}")
print("- Caveats: (i) the split is fixed, so seed variance covers initialisation/shuffling only, not patient sampling; "
      "(ii) rare classes have very few patients, so validation/test metrics rest on a handful of individuals and classes absent "
      "from a split are excluded from macro averages; (iii) no statistical significance test was run, so differences are descriptive only.")
print("\nAll outputs are in:", OUT)

## 19. Zip the Outputs for Download

In [ ]:
# ============================================================
# 19. ZIP THE OUTPUTS FOR DOWNLOAD  (feature cache excluded)
# ============================================================
import zipfile
from pathlib import Path

name = "adamw"
base = Path("/kaggle/working")
src = next((base / n for n in (f"resp_cnn_{name}", f"resp_cnn_{name}_smoke") if (base / n).exists()), None)
assert src is not None, f"Folder not found. Folders: {[p.name for p in base.iterdir()]}"

zip_path = base / f"{name}_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in src.rglob("*"):
        if p.is_file() and "cache" not in p.relative_to(src).parts:
            z.write(p, p.relative_to(src.parent))
print("Source folder:", src.name)
print("Zip size (MB):", round(zip_path.stat().st_size / 1e6, 1))
print("Download: right sidebar -> Output -> /kaggle/working -> refresh -> three dots next to", zip_path.name, "-> Download")